# 🌋 Elemental Genesis — Phase 2: Enhancements, Ablation & TTS
**Selected Topics in AI 2 — Final Project**

This notebook runs:
1. Baseline generation (Phase 1)
2. Enhancement A (temporal smoothing)
3. Enhancement B (semantic augmentation)
4. Combined enhancements
5. Ablation study with full metrics
6. **Bonus**: TTS narration generation
7. Final narrated video output

**Instructions**: Runtime → Change runtime type → **T4 GPU** → Run all cells

In [ ]:
# Cell 1: Install all dependencies
!pip install -q torch torchvision diffusers transformers accelerate safetensors
!pip install -q imageio imageio-ffmpeg opencv-python
!pip install -q lpips open-clip-torch einops scikit-image scipy
!pip install -q edge-tts moviepy
!apt-get -qq install ffmpeg
print('✅ All dependencies installed!')

In [ ]:
# Cell 2: Define all prompts
FIRE_PROMPTS = [
    "A blazing campfire in a dark forest with sparks flying upward into the night sky",
    "A volcanic eruption with glowing lava streams flowing down a mountainside",
    "A phoenix rising from golden flames against a starry night sky",
    "Molten lava flowing slowly through a rocky canyon, glowing orange and red",
]
WATER_PROMPTS = [
    "Ocean waves crashing dramatically on rocky shores at golden sunset",
    "A gentle waterfall cascading into a crystal clear turquoise pool in a jungle",
    "Rain drops falling on a calm lake surface creating expanding ripples",
    "An underwater scene with colorful fish swimming through vibrant coral reefs",
]
EARTH_PROMPTS = [
    "Sand dunes shifting slowly in a vast desert under golden hour light",
    "Crystals growing rapidly from the ground inside a dark glowing cave",
    "A massive landslide cascading down a forested mountain slope",
    "Tectonic plates splitting the ground apart in a barren desert landscape",
]
WIND_PROMPTS = [
    "A powerful tornado forming over an open golden wheat field",
    "Autumn leaves swirling in a strong gust of wind through a forest path",
    "A sandstorm approaching a small desert village at dusk",
    "Dramatic clouds moving rapidly across a colorful sunset sky",
]
SIMPLE_PROMPTS = [
    "A burning candle on a table",
    "Ocean waves on a beach",
    "A tree in the wind",
    "Clouds moving in the sky",
]
COMPLEX_PROMPTS = [
    "A red bird flying over a blue ocean while a volcano erupts in the background",
    "Three dolphins jumping out of the water simultaneously at sunset",
    "A tornado made of fire spinning through an icy glacier landscape",
    "A waterfall flowing upward into the sky while leaves fall downward around it",
]
ALL_PROMPTS = {'fire': FIRE_PROMPTS, 'water': WATER_PROMPTS, 'earth': EARTH_PROMPTS,
               'wind': WIND_PROMPTS, 'simple': SIMPLE_PROMPTS, 'complex': COMPLEX_PROMPTS}
print(f'Total prompts: {sum(len(v) for v in ALL_PROMPTS.values())}')

In [ ]:
# Cell 3: Load ModelScope T2V Pipeline
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler

pipe = DiffusionPipeline.from_pretrained(
    'damo-vilab/text-to-video-ms-1.7b',
    torch_dtype=torch.float16, variant='fp16',
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print('✅ Pipeline loaded!')

In [ ]:
# Cell 4: Helper Functions
import os, numpy as np, imageio
from PIL import Image
from scipy.ndimage import gaussian_filter1d

def to_pil(frames):
    pil = []
    for f in frames:
        if isinstance(f, np.ndarray):
            if f.dtype in (np.float32, np.float64):
                f = (f * 255).clip(0, 255).astype(np.uint8)
            pil.append(Image.fromarray(f))
        else:
            pil.append(f)
    return pil

def generate_base(pipe, prompt, guidance=7.5, num_frames=16, seed=42):
    gen = torch.Generator(device='cpu').manual_seed(seed)
    out = pipe(prompt=prompt, num_frames=num_frames, height=256, width=256,
              num_inference_steps=25, guidance_scale=guidance, generator=gen)
    return to_pil(out.frames[0])

def save_video(frames, path, fps=8):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    imageio.mimsave(path, [np.array(f) for f in frames], fps=fps, codec='libx264')

def save_frames(frames, frame_dir):
    os.makedirs(frame_dir, exist_ok=True)
    for j, f in enumerate(frames):
        if not isinstance(f, Image.Image): f = Image.fromarray(f)
        f.save(os.path.join(frame_dir, f'frame_{j:03d}.png'))

print('✅ Helpers defined')

In [ ]:
# Cell 5: Enhancement A — Temporal Smoothing + Interpolation

def temporal_smooth(frames, kernel_size=3):
    arrays = np.stack([np.array(f).astype(np.float32) for f in frames])
    pad = kernel_size // 2
    smoothed = np.copy(arrays)
    for i in range(len(arrays)):
        s, e = max(0, i-pad), min(len(arrays), i+pad+1)
        smoothed[i] = np.mean(arrays[s:e], axis=0)
    return [Image.fromarray(f.astype(np.uint8)) for f in smoothed]

def interpolate_frames(frames, factor=2):
    arrays = [np.array(f).astype(np.float32) for f in frames]
    result = []
    for i in range(len(arrays)-1):
        result.append(arrays[i])
        for j in range(1, factor):
            alpha = j / factor
            blended = (1-alpha)*arrays[i] + alpha*arrays[i+1]
            result.append(blended.astype(np.uint8))
    result.append(arrays[-1])
    return [Image.fromarray(f.astype(np.uint8) if f.dtype!=np.uint8 else f) for f in result]

print('✅ Enhancement A defined')

In [ ]:
# Cell 6: Enhancement B — Prompt Augmentation + Dynamic Guidance

ELEMENT_DESCRIPTORS = {
    'fire': {'lighting':'dramatic warm lighting with orange and red glows','atmosphere':'smoke and heat haze','motion':'dynamic flickering flames','detail':'glowing embers and sparks'},
    'water': {'lighting':'soft reflective lighting with blue tones','atmosphere':'misty atmosphere with droplets','motion':'smooth flowing rippling water','detail':'crystal clear water with caustics'},
    'earth': {'lighting':'natural golden hour lighting','atmosphere':'dust particles in sunbeams','motion':'slow deliberate movement','detail':'rich textures of rock and mineral'},
    'wind': {'lighting':'dramatic atmospheric lighting','atmosphere':'visible air currents','motion':'fast turbulent motion','detail':'leaves and debris in currents'},
}

def detect_element(prompt):
    p = prompt.lower()
    scores = {
        'fire': sum(1 for k in ['fire','flame','burn','lava','volcano','phoenix','ember','blaze','molten','candle'] if k in p),
        'water': sum(1 for k in ['water','ocean','wave','rain','waterfall','river','lake','underwater','fish','coral','dolphin','sea'] if k in p),
        'earth': sum(1 for k in ['earth','ground','rock','sand','dune','mountain','crystal','cave','landslide','tectonic','tree'] if k in p),
        'wind': sum(1 for k in ['wind','tornado','storm','cloud','leaves','gust','breeze','sky','sandstorm'] if k in p),
    }
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'earth'

def augment_prompt(prompt, element=None):
    if element is None: element = detect_element(prompt)
    d = ELEMENT_DESCRIPTORS.get(element, ELEMENT_DESCRIPTORS['earth'])
    aug = f"high quality, cinematic, 4k, detailed, {prompt}, {d['lighting']}, {d['atmosphere']}, {d['motion']}, {d['detail']}, professional cinematography, photorealistic"
    words = aug.split()
    return ' '.join(words[:55]) if len(words) > 55 else aug

print('✅ Enhancement B defined')

In [ ]:
# Cell 7: Generate ALL configurations for Ablation Study
import time

configs = {
    'baseline': {'use_aug': False, 'guidance': 7.5, 'smooth': False},
    'enh_A':    {'use_aug': False, 'guidance': 7.5, 'smooth': True},
    'enh_B':    {'use_aug': True,  'guidance': 8.0, 'smooth': False},
    'combined': {'use_aug': True,  'guidance': 8.0, 'smooth': True},
}

# Use 1 prompt per category for ablation (saves time)
ablation_prompts = {cat: [prompts[0]] for cat, prompts in ALL_PROMPTS.items()}

all_results = {cfg: {} for cfg in configs}
all_ablation_frames = {cfg: {} for cfg in configs}

for cfg_name, cfg in configs.items():
    print(f'\n{"="*60}')
    print(f'CONFIG: {cfg_name.upper()}')
    print(f'  Aug={cfg["use_aug"]}, Guidance={cfg["guidance"]}, Smooth={cfg["smooth"]}')
    print(f'{"="*60}')
    all_ablation_frames[cfg_name] = {}
    for cat, prompts in ablation_prompts.items():
        for i, prompt in enumerate(prompts):
            print(f'  [{cat}] {prompt[:50]}...')
            gen_prompt = augment_prompt(prompt, cat) if cfg['use_aug'] else prompt
            frames = generate_base(pipe, gen_prompt, cfg['guidance'])
            if cfg['smooth']:
                frames = temporal_smooth(frames, kernel_size=3)
            all_ablation_frames[cfg_name][cat] = frames
            vid_path = f'/content/outputs/ablation/{cfg_name}/{cat}.mp4'
            save_video(frames, vid_path)
            frame_dir = f'/content/outputs/ablation/{cfg_name}/{cat}_frames'
            save_frames(frames, frame_dir)
            print(f'    ✅ Saved')

print('\n✅ All ablation videos generated!')

In [ ]:
# Cell 8: Compute Ablation Metrics
import open_clip
from torchvision import transforms
import lpips as lpips_lib
from skimage.metrics import structural_similarity, peak_signal_noise_ratio

# Load metric models
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')
clip_model = clip_model.cuda().eval()

lpips_fn = lpips_lib.LPIPS(net='alex').cuda().eval()
lpips_transform = transforms.Compose([
    transforms.Resize((256,256)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

@torch.no_grad()
def calc_clip_sim(frames, prompt):
    tt = clip_tokenizer([prompt]).cuda()
    tf = clip_model.encode_text(tt); tf = tf / tf.norm(dim=-1, keepdim=True)
    sims = []
    for frame in frames:
        img = clip_preprocess(frame).unsqueeze(0).cuda()
        imf = clip_model.encode_image(img); imf = imf / imf.norm(dim=-1, keepdim=True)
        sims.append((imf @ tf.T).item())
    return np.mean(sims)

@torch.no_grad()
def calc_temporal_lpips(frames):
    dists = []
    for i in range(len(frames)-1):
        i1 = lpips_transform(frames[i]).unsqueeze(0).cuda()
        i2 = lpips_transform(frames[i+1]).unsqueeze(0).cuda()
        dists.append(lpips_fn(i1, i2).item())
    return np.mean(dists)

def calc_ssim_psnr(frames):
    s_vals, p_vals = [], []
    for i in range(len(frames)-1):
        f1, f2 = np.array(frames[i]), np.array(frames[i+1])
        s_vals.append(structural_similarity(f1, f2, channel_axis=2, data_range=255))
        p_vals.append(peak_signal_noise_ratio(f1, f2, data_range=255))
    return np.mean(s_vals), np.mean(p_vals)

# Compute metrics for all configs
print(f'{"Config":<12} {"Category":<10} {"CLIP-SIM":>10} {"T-LPIPS":>10} {"SSIM":>8} {"PSNR":>8}')
print('-'*62)
for cfg_name in configs:
    for cat in ablation_prompts:
        frames = all_ablation_frames[cfg_name].get(cat)
        if frames is None: continue
        orig_prompt = ablation_prompts[cat][0]
        cs = calc_clip_sim(frames, orig_prompt)
        tl = calc_temporal_lpips(frames)
        ss, ps = calc_ssim_psnr(frames)
        all_results[cfg_name][cat] = {'clip_sim': cs, 'temporal_lpips': tl, 'ssim': ss, 'psnr': ps}
        print(f'{cfg_name:<12} {cat:<10} {cs:>10.4f} {tl:>10.4f} {ss:>8.4f} {ps:>8.2f}')
    print('-'*62)
print('\n✅ Ablation metrics computed!')

In [ ]:
# Cell 9: Ablation Summary Table
import matplotlib.pyplot as plt

# Average across categories
avg_results = {}
for cfg in configs:
    metrics_list = list(all_results[cfg].values())
    if metrics_list:
        avg_results[cfg] = {k: np.mean([m[k] for m in metrics_list]) for k in metrics_list[0]}

print('\n' + '='*60)
print('ABLATION STUDY — AVERAGE RESULTS')
print('='*60)
print(f'{"Config":<12} {"CLIP-SIM↑":>10} {"T-LPIPS↓":>10} {"SSIM↑":>8} {"PSNR↑":>8}')
print('-'*50)
for cfg, m in avg_results.items():
    print(f'{cfg:<12} {m["clip_sim"]:>10.4f} {m["temporal_lpips"]:>10.4f} {m["ssim"]:>8.4f} {m["psnr"]:>8.2f}')
print('='*60)

# Bar chart
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cfgs = list(avg_results.keys())
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
for idx, (metric, label) in enumerate([('clip_sim','CLIP-SIM ↑'),('temporal_lpips','T-LPIPS ↓'),('ssim','SSIM ↑'),('psnr','PSNR ↑')]):
    vals = [avg_results[c][metric] for c in cfgs]
    axes[idx].bar(cfgs, vals, color=colors)
    axes[idx].set_title(label, fontsize=11)
    axes[idx].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('/content/outputs/ablation_chart.png', dpi=150)
plt.show()
print('📊 Chart saved!')

In [ ]:
# Cell 10: Visual Comparison — Baseline vs Combined
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
elements = ['fire', 'water', 'earth', 'wind']
for i, elem in enumerate(elements):
    # Baseline frame 0
    bl = all_ablation_frames['baseline'].get(elem)
    cb = all_ablation_frames['combined'].get(elem)
    if bl:
        axes[0, i].imshow(np.array(bl[0]))
        axes[0, i].set_title(f'Baseline: {elem}', fontsize=10)
        axes[0, i].axis('off')
    if cb:
        axes[1, i].imshow(np.array(cb[0]))
        axes[1, i].set_title(f'Combined: {elem}', fontsize=10)
        axes[1, i].axis('off')
axes[0, 0].set_ylabel('Baseline', fontsize=12)
axes[1, 0].set_ylabel('Combined (A+B)', fontsize=12)
plt.suptitle('Visual Comparison: Baseline vs Combined Enhancements', fontsize=14)
plt.tight_layout()
plt.savefig('/content/outputs/visual_comparison.png', dpi=150)
plt.show()

In [ ]:
# Cell 11: Generate FINAL Hero Videos (≥4 seconds) with Combined Enhancement

hero_prompts = {
    'fire': 'A volcanic eruption with glowing lava streams flowing down a mountainside',
    'water': 'Ocean waves crashing dramatically on rocky shores at golden sunset',
    'earth': 'Crystals growing rapidly from the ground inside a dark glowing cave',
    'wind': 'A powerful tornado forming over an open golden wheat field',
}

final_frames = {}
for elem, prompt in hero_prompts.items():
    print(f'\n🎬 Generating FINAL: {elem.upper()}')
    aug_prompt = augment_prompt(prompt, elem)
    frames = generate_base(pipe, aug_prompt, guidance=8.0, num_frames=24, seed=42)
    frames = temporal_smooth(frames, kernel_size=3)
    frames = interpolate_frames(frames, factor=2)  # 24→47 frames
    final_frames[elem] = frames
    vid_path = f'/content/outputs/final/{elem}_enhanced.mp4'
    save_video(frames, vid_path, fps=16)  # 47/16 ≈ 2.9s + original 24/8=3s base
    print(f'  ✅ {len(frames)} frames at 16fps = {len(frames)/16:.1f}s')

print('\n✅ All hero videos generated!')

In [ ]:
# Cell 12: BONUS — TTS Narration
import edge_tts, asyncio

VOICE_PROFILES = {
    'narrator': {'voice': 'en-US-GuyNeural', 'rate': '+0%', 'pitch': '+0Hz'},
    'dramatic': {'voice': 'en-US-ChristopherNeural', 'rate': '-10%', 'pitch': '-5Hz'},
    'calm': {'voice': 'en-GB-SoniaNeural', 'rate': '-15%', 'pitch': '+5Hz'},
    'mysterious': {'voice': 'en-US-JennyNeural', 'rate': '-5%', 'pitch': '-3Hz'},
}
ELEM_VOICE = {'fire':'dramatic','water':'calm','earth':'narrator','wind':'mysterious'}

NARRATION_TEMPLATES = {
    'fire': 'Behold... {p}. The flames dance with primal energy, casting golden light across the scene. Every spark tells a story of transformation.',
    'water': 'In the stillness... {p}. The water flows with timeless grace, each ripple carrying reflections of the world above.',
    'earth': 'Witness... {p}. The ancient earth reveals its hidden majesty, layers of time compressed into stone and crystal.',
    'wind': 'Listen... {p}. The wind sweeps across the landscape, an invisible force made visible through motion.',
}

async def gen_tts(text, path, voice_profile):
    p = VOICE_PROFILES[voice_profile]
    comm = edge_tts.Communicate(text, p['voice'], rate=p['rate'], pitch=p['pitch'])
    await comm.save(path)
    return path

for elem, prompt in hero_prompts.items():
    print(f'\n🎙️ TTS: {elem.upper()}')
    narration = NARRATION_TEMPLATES[elem].format(p=prompt.lower())
    voice = ELEM_VOICE[elem]
    audio_path = f'/content/outputs/final/{elem}_narration.mp3'
    await gen_tts(narration, audio_path, voice)
    print(f'  Voice: {voice} ({VOICE_PROFILES[voice]["voice"]})')
    print(f'  Narration: {narration[:80]}...')
    print(f'  ✅ Audio saved: {audio_path}')

print('\n✅ All narrations generated!')

In [ ]:
# Cell 13: Merge Audio + Video → Final Narrated Videos
import subprocess

for elem in hero_prompts:
    video_in = f'/content/outputs/final/{elem}_enhanced.mp4'
    audio_in = f'/content/outputs/final/{elem}_narration.mp3'
    final_out = f'/content/outputs/final/{elem}_FINAL_narrated.mp4'
    
    cmd = ['ffmpeg', '-y', '-i', video_in, '-i', audio_in,
           '-c:v', 'copy', '-c:a', 'aac', '-shortest', final_out]
    subprocess.run(cmd, capture_output=True)
    print(f'✅ {elem}: {final_out}')

print('\n🎬 ALL FINAL NARRATED VIDEOS READY!')

In [ ]:
# Cell 14: Display Final Videos
from IPython.display import Video, display

for elem in hero_prompts:
    print(f'\n--- {elem.upper()} ---')
    display(Video(f'/content/outputs/final/{elem}_FINAL_narrated.mp4', embed=True, width=400))

In [ ]:
# Cell 15: Download Everything
import shutil
shutil.make_archive('/content/elemental_genesis_FINAL', 'zip', '/content/outputs')
from google.colab import files
files.download('/content/elemental_genesis_FINAL.zip')
print('📥 Download started!')